In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy import stats
import os
import pickle
from pathlib import Path

# ── Presentation style — applies to all figures ──────────────────────────────
plt.rcParams.update({
    "font.size":        20,
    "axes.titlesize":   20,
    "axes.labelsize":   20,
    "xtick.labelsize":  20,
    "ytick.labelsize":  20,
    "legend.fontsize":  20,
    "figure.titlesize": 25,
})
IOU_03_COLOR = "royalblue"   # IoU = 0.3 reference lines (current threshold)
IOU_05_COLOR = "red"         # IoU = 0.5 reference lines (strict gate)


# Group-level coregistration quality
Loads `confidence_metrics_Round*.pkl` for every mouse (both default and strict thresholds)
and produces publication-quality summary figures.

**Metrics assessed**
- Mean IoU per component (spatial footprint overlap)
- IoU std per component (match stability across sessions)
- Mean centroid distance per component (spatial drift, pixels)
- Fraction of components matched in ≥ 2 sessions
- Session-by-session IoU (longitudinal stability)
- Default (thresh_cost = 0.7, IoU > 0.3) vs strict (thresh_cost = 0.5, IoU > 0.5)

In [ ]:
# ── Global defaults (override per-mouse with the same key in MOUSE_REGISTRY) ──
DEFAULT_HARD_DISK = "Seagate3"
DEFAULT_CAIMAN_V  = "caiman_final"
STRICT_THRESH     = 0.5

# ── Mouse registry ─────────────────────────────────────────────────────────────
# Required keys : mouse_id, round, task
# Optional keys : hard_disk (default: DEFAULT_HARD_DISK)
#                 caiman_v  (default: DEFAULT_CAIMAN_V)
#                 young     (default: True  → path includes /YOUNG_MICE/)
#                 label     (default: auto  → "Mouse{id}_{task}_R{round}")

MOUSE_REGISTRY = [
    dict(mouse_id=945,  round=1, task="Linear",  young=True),
    dict(mouse_id=945,  round=1.5, task="Linear",  young=True),
    dict(mouse_id=945,  round=1.5, task="TMaze",  young=True),
    dict(mouse_id=946,  round=1, task="Linear",  young=True),
    dict(mouse_id=946,  round=1, task="Linear",  young=True),
    dict(mouse_id=946,  round=1.5, task="TMaze",  young=True),
    dict(mouse_id=943,  round=1,   task="Linear", young=True),
    dict(mouse_id=943,  round=1.5,   task="Linear", young=True),
    dict(mouse_id=943,  round=1.5,   task="TMaze", young=True),
    # dict(mouse_id=847,  round=1, task="Linear",  young=True),
    dict(mouse_id=847,  round=1.5, task="Linear",  young=True),
    dict(mouse_id=847,  round=1.5, task="TMaze",  young=True),
    dict(mouse_id=163,  round=1,   task="Linear", young=False, hard_disk="SeagateClean"),
    dict(mouse_id=163,  round=1.5,   task="TMaze", young=False, hard_disk="SeagateClean"),
    dict(mouse_id=163,  round=1.5,   task="Linear", young=False, hard_disk="SeagateClean"),
    dict(mouse_id=1637, round=1.5,   task="TMaze",  young=False, hard_disk="SeagateClean"),
    dict(mouse_id=1636, round=1.5,   task="Linear",  young=False, hard_disk="SeagateClean"),
    #dict(mouse_id=1636, round=1,   task="Linear",  young=False, hard_disk="SeagateClean"),
    dict(mouse_id=1636, round=1.5,   task="TMaze",  young=False, hard_disk="SeagateClean"),
]

# ── Path builder ───────────────────────────────────────────────────────────────
def _build_entry(entry):
    """Expand one MOUSE_REGISTRY dict into (label, base_dir, round_val)."""
    mid    = entry["mouse_id"]
    rnd    = entry["round"]
    task   = entry["task"]
    hd     = entry.get("hard_disk", DEFAULT_HARD_DISK)
    cv     = entry.get("caiman_v",  DEFAULT_CAIMAN_V)
    young  = entry.get("young",     True)
    label  = entry.get("label",     f"Mouse{mid}_{task}_R{rnd}")

    subdir = "/YOUNG_MICE" if young else ""
    base   = (f"/media/toor/{hd}{subdir}/Mouse{mid}"
              f"/Round{rnd}/{task}/{cv}_Registration_Round{rnd}")
    return label, base, rnd

STRICT_SUBDIR = f"stricter_coregistration_thresh_cost_{str(STRICT_THRESH).replace('.', '_')}"

In [ ]:
def load_mouse(label, base_dir, round_val, strict_subdir):
    """Load confidence DataFrames and raw arrays for one mouse/session set."""
    base   = Path(base_dir)
    strict = base / strict_subdir

    def _load_df(d):
        for ext, reader in [(".pkl", pd.read_pickle), (".csv", pd.read_csv)]:
            p = d / f"confidence_metrics_Round{round_val}{ext}"
            if p.exists():
                return reader(p)
        raise FileNotFoundError(f"No confidence_metrics file in {d}")

    def _load_npy(d, name):
        p = d / f"{name}_Round{round_val}.npy"
        return np.load(p, allow_pickle=False) if p.exists() else None

    return dict(
        label       = label,
        round       = round_val,
        df          = _load_df(base),
        df_strict   = _load_df(strict) if strict.exists() else None,
        iou_scores  = _load_npy(base, "iou_scores"),
        assignments = _load_npy(base, "assignments"),
        base_dir    = base,
    )

all_mice = []
for entry in MOUSE_REGISTRY:
    label, base_dir, rnd = _build_entry(entry)
    try:
        m = load_mouse(label, base_dir, rnd, STRICT_SUBDIR)
        all_mice.append(m)
        n_s = len(m["df_strict"]) if m["df_strict"] is not None else "—"
        print(f"  {label:35s}  {len(m['df']):>5} components (default)   {str(n_s):>5} (strict)")
    except Exception as e:
        print(f"  {label:35s}  FAILED- {e}")

print(f"Loaded {len(all_mice)} / {len(MOUSE_REGISTRY)} mice")

In [ ]:
rows = []
for m in all_mice:
    df = m['df']
    valid = df['mean_iou'].notna()
    df_v  = df[valid]

    row = {
        'Mouse':                      m['label'],
        'N components':               len(df),
        'N matched (≥2 sessions)':    valid.sum(),
        'Fraction matched':           f"{valid.mean()*100:.1f}%",
        'Mean IoU':                   f"{df_v['mean_iou'].mean():.3f}",
        'IoU std (stability)':        f"{df_v['iou_std'].mean():.3f}",
        'Mean centroid dist (px)':    f"{df_v['mean_centroid_dist'].mean():.2f}",
        'Fraction IoU > 0.5':         f"{(df_v['mean_iou'] > 0.5).mean()*100:.1f}%",
        'Fraction centroid < 3 px':   f"{(df_v['mean_centroid_dist'] < 3).mean()*100:.1f}%",
        'High confidence (IoU>0.5 & dist<3px)': f"{((df_v['mean_iou']>0.5) & (df_v['mean_centroid_dist']<3)).mean()*100:.1f}%",
    }

    if m['df_strict'] is not None:
        ds = m['df_strict']
        ds_v = ds[ds['mean_iou'].notna()]
        row['N strict (IoU>0.5)'] = len(ds)
        row['Strict mean IoU']    = f"{ds_v['mean_iou'].mean():.3f}"

    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Mouse')
print(summary_df.T.to_string())


In [ ]:
# Figure 1: per-mouse violin plots — IoU and centroid distance
n = len(all_mice)
fig, axes = plt.subplots(1, 2, figsize=(max(10, n * 3 + 2), 7))

labels    = [m['label'] for m in all_mice]
iou_data  = [m['df']['mean_iou'].dropna().values for m in all_mice]
dist_data = [m['df']['mean_centroid_dist'].dropna().values for m in all_mice]

DARK_MICE = {'163', '1636', '1637'}
def is_dark(label):
    return any(f'Mouse{mid}_' in str(label) for mid in DARK_MICE)

IOU_COL_LIGHT  = '#4c87c4'
IOU_COL_DARK   = '#1a3d6e'
DIST_COL_LIGHT = '#6a5acd'
DIST_COL_DARK  = '#2a1052'

# IoU violin
ax = axes[0]
parts = ax.violinplot(iou_data, showmedians=True, showextrema=False)
for pc, lbl in zip(parts['bodies'], labels):
    pc.set_facecolor(IOU_COL_DARK if is_dark(lbl) else IOU_COL_LIGHT)
    pc.set_alpha(0.85)
parts['cmedians'].set_color('navy'); parts['cmedians'].set_linewidth(2.5)
ax.axhline(0.5, color=IOU_05_COLOR, linestyle='--', linewidth=2, label='IoU = 0.5')
ax.axhline(0.3, color=IOU_03_COLOR, linestyle='--', linewidth=2, label='IoU = 0.3 (current threshold)')
for i, (d, lbl) in enumerate(zip(iou_data, labels)):
    col = IOU_COL_DARK if is_dark(lbl) else IOU_COL_LIGHT
    ax.scatter(np.full(len(d), i+1) + np.random.uniform(-0.08, 0.08, len(d)),
               d, s=3, alpha=0.2, color=col)
ax.set_xticks(range(1, n+1))
ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylabel('Mean IoU per component')
ax.set_title('Spatial footprint overlap (IoU)')
ax.legend()
ax.set_ylim(0, 1)

# Centroid distance violin
ax = axes[1]
parts = ax.violinplot(dist_data, showmedians=True, showextrema=False)
for pc, lbl in zip(parts['bodies'], labels):
    pc.set_facecolor(DIST_COL_DARK if is_dark(lbl) else DIST_COL_LIGHT)
    pc.set_alpha(0.85)
parts['cmedians'].set_color('#2d0b6e'); parts['cmedians'].set_linewidth(2.5)
ax.axhline(10, color='grey', linestyle='--', linewidth=2,
           label=' current max (10 px)')
for i, (d, lbl) in enumerate(zip(dist_data, labels)):
    col = DIST_COL_DARK if is_dark(lbl) else DIST_COL_LIGHT
    ax.scatter(np.full(len(d), i+1) + np.random.uniform(-0.08, 0.08, len(d)),
               d, s=3, alpha=0.2, color=col)
ax.set_xticks(range(1, n+1))
ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylabel('Mean centroid distance (px)')
ax.set_title('Centroid displacement (px)')
ax.legend()

plt.suptitle('Per-mouse coregistration quality', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("group_coregistration_distributions.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: group-level bar chart with individual mouse dots and SEM
def _mouse_means(key, threshold=None, above=True):
    out = []
    for m in all_mice:
        v = m['df'][key].dropna().values
        if threshold is not None:
            v = (v > threshold).mean() * 100 if above else (v < threshold).mean() * 100
        else:
            v = v.mean()
        out.append(v)
    return np.array(out)

metrics = [
    ('mean_iou',           None,  None,   'Mean IoU',                (0, 1)),
    ('iou_std',            None,  None,   'Mean IoU std (stability)', (0, 0.4)),
    ('mean_centroid_dist', None,  None,   'Mean centroid dist (px)',  None),
    ('mean_iou',           0.5,   True,   '% IoU > 0.5',             (0, 100)),
    ('mean_centroid_dist', 3,     False,  '% centroid < 3 px',      (0, 100)),
]

fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 7))
colors = ['#4c87c4', '#6a5acd', '#c4724c', '#4cb87c', '#c4a44c']

for ax, (key, thr, above, ylabel, ylim), col in zip(axes, metrics, colors):
    vals = _mouse_means(key, thr, above) if thr is not None else _mouse_means(key)
    mu, sem = vals.mean(), stats.sem(vals)

    ax.bar([0], [mu], yerr=[sem], capsize=8, color=col, alpha=0.8,
           error_kw=dict(linewidth=2.5))
    ax.scatter(np.zeros(len(vals)) + np.random.uniform(-0.15, 0.15, len(vals)),
               vals, color='black', s=60, zorder=5)

    ax.set_xticks([])
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_xlim(-0.5, 0.5)
    ax.text(0, mu + sem + (ylim[1] * 0.03 if ylim else 0.02),
            f"{mu:.2f}\n\u00b1{sem:.2f}", ha='center', va='bottom')

plt.suptitle('Group coregistration quality  (mean \u00b1 SEM across mice)', fontweight='bold')
plt.tight_layout()
plt.savefig("group_coregistration_summary.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Figure 3: default vs strict threshold comparison
mice_with_strict   = [m for m in all_mice if m['df_strict'] is not None]
mice_without_strict = [m['label'] for m in all_mice if m['df_strict'] is None]
if mice_without_strict:
    print(f"Note: {len(mice_without_strict)} mouse/mice excluded (no strict data): {mice_without_strict}")
if not mice_with_strict:
    print("No strict coregistration data found — run the strict cell in the registration notebook first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    labels_s = [m['label'] for m in mice_with_strict]
    x = np.arange(len(mice_with_strict))
    width = 0.35

    # Panel 1: N components retained
    ax = axes[0]
    n_def    = [len(m['df']) for m in mice_with_strict]
    n_strict = [len(m['df_strict']) for m in mice_with_strict]
    ax.bar(x - width/2, n_def,    width, label='Default  (IoU>0.3)', color=IOU_03_COLOR, alpha=0.85)
    ax.bar(x + width/2, n_strict, width, label='Strict   (IoU>0.5)', color=IOU_05_COLOR, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_s, rotation=20, ha='right')
    ax.set_ylabel('Union components')
    ax.set_title('Components retained')
    ax.legend()

    # Panel 2: mean IoU
    ax = axes[1]
    iou_def_m    = [m['df']['mean_iou'].dropna().mean() for m in mice_with_strict]
    iou_strict_m = [m['df_strict']['mean_iou'].dropna().mean() for m in mice_with_strict]
    ax.bar(x - width/2, iou_def_m,    width, color=IOU_03_COLOR, alpha=0.85, label='Default (IoU>0.3)')
    ax.bar(x + width/2, iou_strict_m, width, color=IOU_05_COLOR, alpha=0.85, label='Strict  (IoU>0.5)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels_s, rotation=20, ha='right')
    ax.set_ylabel('Mean IoU')
    ax.set_ylim(0, 1)
    ax.set_title('Mean IoU per component')
    ax.legend()

    # Panel 3: mean centroid distance
    ax = axes[2]
    cd_def_m    = [m['df']['mean_centroid_dist'].dropna().mean() for m in mice_with_strict]
    cd_strict_m = [m['df_strict']['mean_centroid_dist'].dropna().mean() for m in mice_with_strict]
    ax.bar(x - width/2, cd_def_m,    width, color=IOU_03_COLOR, alpha=0.85, label='Default (IoU>0.3)')
    ax.bar(x + width/2, cd_strict_m, width, color=IOU_05_COLOR, alpha=0.85, label='Strict  (IoU>0.5)')
    ax.set_xticks(x)
    ax.set_xticklabels(labels_s, rotation=20, ha='right')
    ax.set_ylabel('Mean centroid distance (px)')
    ax.set_title('Centroid distance per component')
    ax.legend()

    plt.suptitle('Default vs strict coregistration threshold', fontweight='bold')
    plt.tight_layout()
    plt.savefig("group_coregistration_default_vs_strict.png", dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
from matplotlib.ticker import MaxNLocator
import matplotlib.patches as mpatches

OLD_MICE = {'163', '1636', '1637'}
def is_old(label):
    return any(f'Mouse{mid}_' in str(label) for mid in OLD_MICE)

YOUNG_COL = '#4c87c4'
OLD_COL   = '#1a3d6e'

# Figure 4: longitudinal stability — IoU per session index
fig, ax = plt.subplots(figsize=(13, 8))

for m in all_mice:
    if m['iou_scores'] is None:
        print(f"  {m['label']}: no iou_scores.npy found, skipping")
        continue

    iou    = m['iou_scores']
    n_sess = iou.shape[1]

    sess_median = np.array([np.nanmedian(iou[:, j]) for j in range(n_sess)])
    valid_sess  = ~np.isnan(sess_median)
    x = np.where(valid_sess)[0]

    col = OLD_COL if is_old(m['label']) else YOUNG_COL
    ax.plot(x, sess_median[valid_sess], '-o', color=col, linewidth=2.5,
            markersize=7, alpha=0.7)

ax.axhline(0.5, color=IOU_05_COLOR, linestyle='--', linewidth=2)
ax.axhline(0.3, color=IOU_03_COLOR, linestyle='--', linewidth=2)

ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlabel('Session index')
ax.set_ylabel('Median IoU')
ax.set_title('Coregistration stability')
ax.legend(handles=[
    mpatches.Patch(color=YOUNG_COL, label='Young mice'),
    mpatches.Patch(color=OLD_COL,   label='Old mice'),
    plt.Line2D([0], [0], color=IOU_05_COLOR, linestyle='--', linewidth=2, label='IoU = 0.5'),
    plt.Line2D([0], [0], color=IOU_03_COLOR, linestyle='--', linewidth=2, label='IoU = 0.3 (current threshold)'),
], bbox_to_anchor=(1.01, 1), loc='best')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("group_coregistration_longitudinal.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Printable group-level statistics for methods/results section
print("=" * 60)
print("GROUP COREGISTRATION QUALITY — summary statistics")
print("=" * 60)

all_iou   = np.concatenate([m['df']['mean_iou'].dropna().values for m in all_mice])
all_cdist = np.concatenate([m['df']['mean_centroid_dist'].dropna().values for m in all_mice])
all_istd  = np.concatenate([m['df']['iou_std'].dropna().values for m in all_mice])
all_nmat  = np.concatenate([m['df']['n_sessions_matched'].values for m in all_mice])

n_mice   = len(all_mice)
n_total  = sum(len(m['df']) for m in all_mice)
n_matched = sum((m['df']['mean_iou'].notna()).sum() for m in all_mice)

print(f"\nMice: {n_mice}  |  Total union components: {n_total}  |  Matched in ≥2 sessions: {n_matched} ({n_matched/n_total*100:.1f}%)")

print(f"\n--- Spatial footprint overlap (IoU) ---")
print(f"  Mean ± SD         : {all_iou.mean():.3f} ± {all_iou.std():.3f}")
print(f"  Median [IQR]      : {np.median(all_iou):.3f}  [{np.percentile(all_iou,25):.3f} – {np.percentile(all_iou,75):.3f}]")
print(f"  IoU > 0.5         : {(all_iou > 0.5).mean()*100:.1f}% of matched components")
print(f"  IoU < 0.3         : {(all_iou < 0.3).mean()*100:.1f}% of matched components")

print(f"\n--- IoU stability (std across sessions) ---")
print(f"  Mean ± SD         : {all_istd.mean():.3f} ± {all_istd.std():.3f}  (lower = more stable)")

print(f"\n--- Centroid distance (pixels, post-alignment) ---")
print(f"  Mean ± SD         : {all_cdist.mean():.2f} ± {all_cdist.std():.2f} px")
print(f"  Median [IQR]      : {np.median(all_cdist):.2f}  [{np.percentile(all_cdist,25):.2f} – {np.percentile(all_cdist,75):.2f}] px")
print(f"  < 2 px            : {(all_cdist < 2).mean()*100:.1f}%")
print(f"  < 3 px            : {(all_cdist < 3).mean()*100:.1f}%")
print(f"  > 5 px            : {(all_cdist > 5).mean()*100:.1f}%")

print(f"\n--- High-confidence coregistration (IoU>0.5 AND centroid<3px) ---")
all_df = pd.concat([m['df'] for m in all_mice], ignore_index=True)
high_conf = all_df[(all_df['mean_iou'] > 0.5) & (all_df['mean_centroid_dist'] < 3)]
print(f"  N components      : {len(high_conf)} / {n_matched} matched  ({len(high_conf)/n_matched*100:.1f}%)")

if any(m['df_strict'] is not None for m in all_mice):
    s_mice = [m for m in all_mice if m['df_strict'] is not None]
    s_all  = pd.concat([m['df_strict'] for m in s_mice], ignore_index=True)
    s_v    = s_all['mean_iou'].dropna()
    print(f"\n--- Strict threshold (IoU>0.5 gate) ---")
    print(f"  N retained        : {len(s_all)} (vs {sum(len(m['df']) for m in s_mice)} default)")
    print(f"  Mean IoU (strict) : {s_v.mean():.3f} ± {s_v.std():.3f}")
print("=" * 60)


---
### Soma radius estimation from spatial footprints

For each union component, the area in pixels is:
`area_px = number of pixels above 10% of the component's peak weight`

Assuming a circular soma approximation: `radius_px = sqrt(area_px / π)`

For hippocampal CA1 pyramidal cells the expected soma diameter is ~15–25 µm.
Comparing the pixel-space radius against this gives an empirical µm/px calibration
and a biologically grounded centroid distance threshold (a shift > 1 radius = >half a soma diameter).

In [ ]:
import math
from scipy.optimize import brentq

# ── IoU → centroid shift for two equal circles (exact formula) ────────────────
def circle_iou(d_over_r):
    d = d_over_r
    if d >= 2.0: return 0.0
    if d == 0.0: return 1.0
    A_int = 2*math.acos(d/2) - (d/2)*math.sqrt(4 - d**2)  # normalised r=1
    return A_int / (2*math.pi - A_int)

def shift_for_iou(iou_threshold):
    """Centroid shift (in units of r) at which IoU equals iou_threshold."""
    return brentq(lambda d: circle_iou(d) - iou_threshold, 0, 1.999)

DEFAULT_THRESH = 0.7   # thresh_cost default → IoU > 0.3
STRICT_THRESH  = 0.5   # strict              → IoU > 0.5

d_default = shift_for_iou(1 - DEFAULT_THRESH)   # 0.875 r
d_strict  = shift_for_iou(1 - STRICT_THRESH)    # 0.530 r

print(f"IoU thresholds as centroid shift (units of soma radius r):")
print(f"  Default (thresh_cost={DEFAULT_THRESH}, IoU>{1-DEFAULT_THRESH:.1f}): max shift = {d_default:.3f} r")
print(f"  Strict  (thresh_cost={STRICT_THRESH}, IoU>{1-STRICT_THRESH:.1f}): max shift = {d_strict:.3f} r")

# ── Load footprints and compute radius in pixels ──────────────────────────────
mouse_radii = {}
mouse_areas = {}

for m in all_mice:
    su_path = m["base_dir"] / f"spatial_union_Round{m['round']}.npy"
    if not su_path.exists():
        print(f"  {m['label']}: spatial_union not found — skipping")
        continue

    print(f"  {m['label']}: loading...", end=" ", flush=True)
    su = np.load(su_path)

    col_max  = su.max(axis=0)
    valid    = col_max > 0
    areas_px = np.zeros(su.shape[1])
    areas_px[valid] = (su[:, valid] > 0.1 * col_max[valid]).sum(axis=0)

    areas_valid = areas_px[areas_px > 0]
    radii       = np.sqrt(areas_valid / np.pi)

    mouse_radii[m["label"]] = radii
    mouse_areas[m["label"]] = areas_valid

    print(f"{len(radii)} components | "
          f"mean radius = {radii.mean():.1f} ± {radii.std():.1f} px | "
          f"median = {np.median(radii):.1f} px")
    del su

# ── Pooled summary ────────────────────────────────────────────────────────────
all_radii = np.concatenate(list(mouse_radii.values()))
mean_r    = all_radii.mean()
std_r     = all_radii.std()
med_r     = np.median(all_radii)

# Centroid thresholds in pixels — derived from data, no µm needed
d_default_px = d_default * mean_r
d_strict_px  = d_strict  * mean_r

print(f"\nPooled ({len(all_radii)} components, {len(mouse_radii)} mice):")
print(f"  Mean radius          : {mean_r:.1f} ± {std_r:.1f} px")
print(f"  Median radius        : {med_r:.1f} px")
print(f"  Mean diameter        : {2*mean_r:.1f} px")
print(f"\n  Max centroid shift allowed by each IoU threshold:")
print(f"    Default (IoU>0.3)  : {d_default:.3f} × {mean_r:.1f} px = {d_default_px:.1f} px")
print(f"    Strict  (IoU>0.5)  : {d_strict:.3f} × {mean_r:.1f} px = {d_strict_px:.1f} px")

# ── Optional µm calibration (set SOMA_AREA_UM2 = None to skip) ───────────────
SOMA_AREA_UM2 = 137.0
if SOMA_AREA_UM2 is not None:
    mean_area_px = np.concatenate(list(mouse_areas.values())).mean()
    um_per_px    = math.sqrt(SOMA_AREA_UM2 / mean_area_px)
    print(f"\n  µm calibration (from {SOMA_AREA_UM2} µm² soma area):")
    print(f"    Pixel size          : {um_per_px:.3f} µm/px")
    print(f"    Mean radius         : {mean_r * um_per_px:.1f} µm")
    print(f"    Max shift (default) : {d_default_px * um_per_px:.1f} µm")
    print(f"    Max shift (strict)  : {d_strict_px * um_per_px:.1f} µm")
else:
    um_per_px = None

In [ ]:
if not mouse_radii:
    print("No radius data - run the computation cell above first.")
else:
    FS_TITLE  = 20
    FS_LABEL  = 20
    FS_TICK   = 15
    FS_LEGEND = 15

    plt.rcParams.update({
        "font.size":        FS_TICK,
        "axes.titlesize":   FS_TITLE,
        "axes.labelsize":   FS_LABEL,
        "xtick.labelsize":  FS_TICK,
        "ytick.labelsize":  FS_TICK,
        "legend.fontsize":  FS_LEGEND,
    })

    SOMA_YOUNG_COL = '#e07b54'
    SOMA_OLD_COL   = '#7a3010'

    labels_r    = list(mouse_radii.keys())
    radius_data = [mouse_radii[l] for l in labels_r]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # ── Panel 1: per-mouse violin ─────────────────────────────────────────────
    ax = axes[0]
    parts = ax.violinplot(radius_data, showmedians=True, showextrema=False)
    for pc, lbl in zip(parts["bodies"], labels_r):
        col = SOMA_OLD_COL if is_old(lbl) else SOMA_YOUNG_COL
        pc.set_facecolor(col); pc.set_alpha(0.85)
    parts["cmedians"].set_color("#3a1508"); parts["cmedians"].set_linewidth(2.5)
    for i, (d, lbl) in enumerate(zip(radius_data, labels_r)):
        col = SOMA_OLD_COL if is_old(lbl) else SOMA_YOUNG_COL
        ax.scatter(np.full(len(d), i+1) + np.random.uniform(-0.08, 0.08, len(d)),
                   d, s=3, alpha=0.15, color=col)

    ax.axhline(mean_r, color="navy", linestyle="--", linewidth=2,
               label=f"Pooled mean = {mean_r:.1f} px")
    ax.axhspan(mean_r - std_r, mean_r + std_r, alpha=0.1, color="navy",
               label=f"±1 SD")
    ax.set_xticks(range(1, len(labels_r)+1))
    ax.set_xticklabels(labels_r, rotation=20, ha="right", fontsize=FS_TICK)
    ax.set_ylabel("Soma radius  (px)", fontsize=FS_LABEL)
    ax.set_ylim(0, 20)
    ax.set_title("Per-mouse x round x task soma radii", fontsize=FS_TITLE)
    ax.legend(fontsize=FS_LEGEND)

    # ── Panel 2: pooled histogram ─────────────────────────────────────────────
    ax = axes[1]
    ax.hist(all_radii, bins=60, color="#e07b54", edgecolor="white",
            linewidth=0.4, density=True, alpha=0.85)
    ax.axvline(mean_r, color="navy", linestyle="--", linewidth=2.5,
               label=f"Mean = {mean_r:.1f} px")

    ax.set_xlabel("Soma radius  (px)", fontsize=FS_LABEL)
    ax.set_xlim(0, 20)
    ax.set_ylabel("Density", fontsize=FS_LABEL)
    ax.set_title(f"soma radii  (n = {len(all_radii)})", fontsize=FS_TITLE)
    plt.tight_layout()
    plt.savefig("group_soma_radius.png", dpi=300, bbox_inches="tight")
    plt.show()

    # reset rcParams so downstream cells aren't affected
    plt.rcdefaults()

    # ── Ready-to-paste update for Figure 1 centroid panel ────────────────────
    print("Update Figure 1 centroid panel — replace the 10 px gate line with:")
    print(f"  ax.axhline({d_default_px:.1f}, color='orange', linestyle='--',")
    print(f"             label='Default IoU gate (IoU>0.3) = {d_default:.2f}r = {d_default_px:.1f} px')")
    print(f"  ax.axhline({d_strict_px:.1f},  color='green',  linestyle='--',")
    print(f"             label='Strict  IoU gate  (IoU>0.5) = {d_strict:.2f}r = {d_strict_px:.1f} px')")